# 01 Tokenizer、移位标签与语言模型损失
本 notebook 只做前向计算，不需要 checkpoint 或 GPU。请从项目根目录启动 Jupyter。

In [ ]:
from pathlib import Path
import sys, torch
ROOT = Path.cwd()
if not (ROOT / 'model').exists(): ROOT = ROOT.parents[2]
sys.path.insert(0, str(ROOT))
from tokenizer.tokenizer_utils import MiniLLMTokenizer
tok = MiniLLMTokenizer(str(ROOT / 'tokenizer/minillm_tokenizer.json'))

In [ ]:
text = 'Once upon a time, a little bird sang.'
ids = tok.encode(text, add_special_tokens=True)
print(ids)
print(tok.decode(ids, skip_special_tokens=True))
print('vocab =', tok.vocab_size)

对序列 $[x_1,\ldots,x_T]$，input 使用前 $T-1$ 项，target 使用后 $T-1$ 项。

In [ ]:
input_ids = torch.tensor(ids[:-1])
targets = torch.tensor(ids[1:])
for x, y in zip(input_ids[:8], targets[:8]):
    print(repr(tok.decode([int(x)])), '->', repr(tok.decode([int(y)])))

In [ ]:
import torch.nn.functional as F
T, V = len(targets), tok.vocab_size
logits = torch.randn(T, V)
loss = F.cross_entropy(logits, targets)
manual = -F.log_softmax(logits, dim=-1).gather(1, targets[:, None]).mean()
print('cross entropy:', loss.item())
print('manual NLL:   ', manual.item())
print('perplexity:   ', loss.exp().item())
assert torch.allclose(loss, manual)